In [ ]:
%pip install -q "ultralytics==8.4.161"
import json, shutil, zipfile, yaml
from pathlib import Path
from ultralytics import YOLO
!rm -rf /content/tmp && git clone -q --depth 1 -b v5-tmp https://github.com/dportilla219/eggs-detector /content/tmp
T = Path('/content/tmp'); D = Path('/content/eggs_v2')
for f in sorted(T.glob('*.parte1de*')):
    nombre, total = f.name.rsplit('.parte1de', 1)
    with open(T / nombre, 'wb') as out:
        for i in range(1, int(total) + 1):
            out.write((T / f'{nombre}.parte{i}de{total}').read_bytes())
if not (D / 'test').exists():
    with zipfile.ZipFile(T / 'eggs_v2_parte4_valid_test.zip') as zf:
        zf.extractall(D)
yaml.safe_dump({'path': str(D), 'train': str(D / 'valid/images'), 'val': str(D / 'valid/images'), 'test': str(D / 'test/images'), 'nc': 2, 'names': ['Crack', 'Intact']}, open(D / 'd.yaml', 'w'))
res = {}
for nombre, pt in [('v2', 'v2_best.pt'), ('v5', 'eggs_v5.pt')]:
    m = YOLO(str(T / pt)).val(data=str(D / 'd.yaml'), split='test', imgsz=640, batch=16, plots=False, verbose=False)
    r = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, rc, ap50, ap = m.box.class_result(i)
        r[['Crack', 'Intact'][c]] = {'P': float(p), 'R': float(rc), 'mAP50': float(ap50), 'mAP50-95': float(ap)}
    res[nombre] = r
print(json.dumps(res, indent=1))
json.dump(res, open('/content/eggs_v5_val.json', 'w'))
from google.colab import files
files.download('/content/eggs_v5_val.json')
